# Cervical Cancer Stage Classification on Kaggle

This notebook launches the backend training script with Kaggle-friendly paths. It looks for the repository, finds the dataset, and writes checkpoints to the Kaggle working directory.

## Kaggle Run Guide

1. Open this notebook in Kaggle.
2. Make sure the repository is available in the session, or allow the notebook to clone it automatically.
3. If your dataset is mounted at a custom path, set `DATA_DIR` in Cell 3.
4. Run the notebook from top to bottom.
5. Check `/kaggle/working/Checkpoints` for `best_model.pt`, `last_model.pt`, `history.json`, and `metrics.json`.

Recommended defaults for the current run are already set in the training cell.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Shubh-Rawat7/Cervical-Cancer-Classifier.git'
REPO_NAME = 'Cervical-Cancer-Classifier'


def find_backend_dir() -> Path | None:
    search_roots = [Path('/kaggle/working'), Path('/kaggle/input'), Path.cwd()]
    direct_candidates = [
        Path('/kaggle/working/backend'),
        Path('/kaggle/input/backend'),
        Path.cwd() / 'backend',
        Path('/kaggle/working') / REPO_NAME / 'backend',
        Path('/kaggle/input') / REPO_NAME / 'backend',
    ]

    for candidate in direct_candidates:
        if (candidate / 'train.py').exists():
            return candidate

    for root in search_roots:
        if not root.exists():
            continue
        for match in root.rglob('train.py'):
            if match.name == 'train.py' and match.parent.name == 'backend':
                return match.parent
    return None


def clone_repo_if_needed() -> Path:
    work_root = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
    repo_dir = work_root / REPO_NAME
    backend_dir = repo_dir / 'backend'
    if (backend_dir / 'train.py').exists():
        return backend_dir

    if repo_dir.exists():
        print(f'Removing incomplete repo folder: {repo_dir}')
        subprocess.run(['rm', '-rf', str(repo_dir)], check=False)

    print(f'Cloning repository from {REPO_URL}')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_dir)], check=True)
    if not (backend_dir / 'train.py').exists():
        raise FileNotFoundError(f'Cloned repo but could not find backend/train.py in {backend_dir}')
    return backend_dir


BACKEND_DIR = find_backend_dir()
if BACKEND_DIR is None:
    BACKEND_DIR = clone_repo_if_needed()

REPO_ROOT = BACKEND_DIR.parent
os.chdir(REPO_ROOT)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print('REPO_ROOT:', REPO_ROOT)
print('BACKEND_DIR:', BACKEND_DIR)
print('Python:', sys.executable)

In [ ]:
CLASS_NAMES = ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer']


def _class_count(base: Path) -> int:
    try:
        return sum(1 for name in CLASS_NAMES if (base / name).is_dir())
    except Exception:
        return 0


def _has_class_folders(base: Path) -> bool:
    return base.exists() and base.is_dir() and _class_count(base) == len(CLASS_NAMES)


def _looks_like_dataset_root(base: Path) -> bool:
    if not base.exists() or not base.is_dir():
        return False

    if _has_class_folders(base):
        return True

    for split_name in ('train', 'val', 'test'):
        split_dir = base / split_name
        if _has_class_folders(split_dir):
            return True
    return False


def find_data_dir() -> Path | None:
    raw_candidates = [
        os.environ.get('DATA_DIR', ''),
        '/kaggle/input/datasets/shubhrawat132/herlevdataset',
        '/kaggle/input/Herlev Dataset',
        '/kaggle/input/herlev-dataset',
        '/kaggle/input/herlevdataset',
        '/kaggle/input/cervical-cancer-stage-classification',
        '/kaggle/input/cervical-cancer-dataset',
        str(REPO_ROOT / 'Herlev Dataset'),
        str(REPO_ROOT / 'data'),
    ]

    for candidate_text in raw_candidates:
        if not candidate_text:
            continue
        candidate = Path(candidate_text)
        if not candidate.exists():
            continue

        # Prefer the highest dataset root, not a split leaf.
        if _looks_like_dataset_root(candidate):
            return candidate

        try:
            for root in candidate.rglob('*'):
                if root.is_dir() and _looks_like_dataset_root(root):
                    return root
        except Exception:
            pass

    return None


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path('/kaggle/working/Checkpoints') if Path('/kaggle/working').exists() else (REPO_ROOT / 'backend' / 'Checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if DATA_DIR is None:
    raise FileNotFoundError('Could not find the dataset. Set DATA_DIR to your Kaggle input folder and rerun this cell.')

print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Class folders found:', {name: (DATA_DIR / name).exists() for name in CLASS_NAMES})

In [ ]:
train_script = BACKEND_DIR / 'train.py'

help_text = subprocess.run([sys.executable, str(train_script), '--help'], capture_output=True, text=True).stdout
supports_new_stage_args = '--stage1-epochs' in help_text and '--cb-beta' in help_text and '--swa-epochs' in help_text
supports_old_phase_args = '--phase1-epochs' in help_text and '--phase2-epochs' in help_text

command = [
    sys.executable,
    str(train_script),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '90',
    '--batch-size', '16',
    '--img-size', '256',
    '--backbone', 'mambavision_small',
    '--dropout', '0.30',
    '--lr-head', '3e-4',
    '--lr-backbone', '3e-5',
    '--weight-decay', '1e-4',
    '--mixup-alpha', '0.30',
    '--patience', '15',
    '--gradient-clip', '1.0',
    '--num-workers', str(min(4, os.cpu_count() or 2)),
]

if supports_new_stage_args:
    command.extend([
        '--stage1-epochs', '24',
        '--stage2-epochs', '26',
        '--stage3-epochs', '40',
        '--cb-beta', '0.9999',
        '--cb-gamma', '2.0',
        '--scheduler-t0', '10',
        '--eta-min', '1e-6',
        '--swa-epochs', '20',
        '--swa-lr', '1e-5',
        '--use-amp',
    ])
elif supports_old_phase_args:
    command.extend([
        '--phase1-epochs', '24',
        '--phase2-epochs', '26',
        '--focal-gamma', '2.0',
        '--use-amp',
    ])
else:
    raise RuntimeError(f'Unsupported train.py CLI. Help output was:\n{help_text}')

print('Running hybrid MambaVision + handcrafted-feature training:')
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
print('Training artifacts written to:', OUTPUT_DIR)
artifacts = sorted(OUTPUT_DIR.glob('*'))
if not artifacts:
    print('No artifacts found yet. Run the training cell first.')
else:
    print('Training artifacts:')
    for artifact in artifacts:
        print('-', artifact.name)

for name in [
    'best_model.pt',
    'last_model.pt',
    'swa_model.pt',
    'history.json',
    'metrics.json',
    'learning_curves.png',
    'confusion_matrix.png',
    'roc_curves.png',
    'precision_recall_curves.png',
    'misclassified_samples.png',
]:
    path = OUTPUT_DIR / name
    print(f'{name}:', path.exists())

metrics_path = OUTPUT_DIR / 'metrics.json'
if metrics_path.exists():
    print('metrics.json saved at', metrics_path)
    print(metrics_path.read_text()[:2000])